# α,β-CROWN

## What This Is
α,β-CROWN is the state-of-the-art neural network verifier (winner of VNN-COMP 2021-2023). It combines:

**CROWN (2018)**: Computes linear bounds on neural network outputs efficiently:
- Lower bound: f(x) ≥ αᵀx + β for all x in the input domain
- Upper bound: f(x) ≤ α̂ᵀx + β̂
These bounds are tighter than interval bound propagation because they carry slope information.

**α relaxation (α-CROWN)**: Makes the lower bound slopes α learnable, optimized via gradient ascent to maximize the lower bound (tightest possible sound approximation).

**β relaxation (β-CROWN)**: Extends to branch-and-bound where β variables encode split decisions, enabling efficient integration of case analysis.

**Branch-and-Bound (BaB)**: Splits the input domain on neurons that are hardest to bound, recursively certifying sub-problems. α,β-CROWN uses GPU-parallelized BaB to handle thousands of neurons.

In [ ]:
import numpy as np
from typing import Tuple, List, Optional

np.random.seed(42)

# CROWN linear bound computation
# For a ReLU network, compute linear lower/upper bounds on each neuron

def crown_bounds(W1, b1, W2, b2, x_lo, x_hi):
    """
    Compute CROWN linear bounds on outputs.
    Returns: (output_lower_bound, output_upper_bound)
    as linear functions of input x (stored as (slope, intercept) for bounds).
    
    Simplified: returns concrete bounds via propagation of linear relaxations.
    """
    n1 = len(b1)  # hidden dim
    
    # Layer 1 pre-activation bounds
    W1p = np.maximum(0, W1); W1n = np.minimum(0, W1)
    pre_lo = W1p @ x_lo + W1n @ x_hi + b1
    pre_hi = W1p @ x_hi + W1n @ x_lo + b1
    
    # CROWN ReLU relaxation:
    # For ambiguous neurons (pre_lo < 0 < pre_hi), use linear relaxation:
    # Lower bound: 0 (take 0 as lower)
    # Upper bound: slope * pre + intercept where slope = pre_hi/(pre_hi-pre_lo)
    crown_lower_slopes = np.zeros(n1)
    crown_upper_slopes = np.zeros(n1)
    crown_upper_offsets = np.zeros(n1)
    
    for i in range(n1):
        lo, hi = pre_lo[i], pre_hi[i]
        if lo >= 0:
            # Active: h = pre
            crown_lower_slopes[i] = 1.0
            crown_upper_slopes[i] = 1.0
        elif hi <= 0:
            # Inactive: h = 0
            pass
        else:
            # Ambiguous: linear relaxation
            slope = hi / (hi - lo)
            crown_lower_slopes[i] = 0.0  # tightest lower: alpha (optimizable)
            crown_upper_slopes[i] = slope
            crown_upper_offsets[i] = -slope * lo
    
    # Propagate through Layer 2 using the relaxation
    # Use IBP on the relaxed hidden layer bounds
    h_lo_crown = crown_lower_slopes * pre_lo  # conservative lower
    h_hi_crown = crown_upper_slopes * pre_hi + crown_upper_offsets
    
    W2p = np.maximum(0, W2); W2n = np.minimum(0, W2)
    out_lo = W2p @ h_lo_crown + W2n @ h_hi_crown + b2
    out_hi = W2p @ h_hi_crown + W2n @ h_lo_crown + b2
    
    return out_lo, out_hi

# Compare IBP vs CROWN bounds
W1 = np.array([[2.0, 1.0, 0.5], [-1.0, 2.0, -0.5], [1.5, -0.5, 1.0]])
b1 = np.array([0.5, 0.5, 0.2])
W2 = np.array([[1.0, 0.5, 0.3], [-1.0, -0.5, -0.3]])
b2 = np.array([0.1, -0.1])

x0 = np.array([0.5, 0.3, 0.2])

for eps in [0.05, 0.1, 0.2]:
    x_lo = x0 - eps; x_hi = x0 + eps
    
    # IBP bounds
    W1p = np.maximum(0, W1); W1n = np.minimum(0, W1)
    pre_lo = W1p @ x_lo + W1n @ x_hi + b1
    pre_hi = W1p @ x_hi + W1n @ x_lo + b1
    h_lo_ibp = np.maximum(0, pre_lo)
    h_hi_ibp = np.maximum(0, pre_hi)
    W2p = np.maximum(0, W2); W2n = np.minimum(0, W2)
    ibp_lo = W2p @ h_lo_ibp + W2n @ h_hi_ibp + b2
    ibp_hi = W2p @ h_hi_ibp + W2n @ h_lo_ibp + b2
    ibp_gap = ibp_hi[0] - ibp_lo[0]
    
    # CROWN bounds
    crown_lo, crown_hi = crown_bounds(W1, b1, W2, b2, x_lo, x_hi)
    crown_gap = crown_hi[0] - crown_lo[0]
    
    print(f'eps={eps}: IBP gap={ibp_gap:.4f}, CROWN gap={crown_gap:.4f}, '
          f'CROWN tighter by {(ibp_gap-crown_gap)/ibp_gap*100:.1f}%')


In [ ]:
# Alpha optimization: tighten CROWN lower bounds
# The alpha parameter controls the lower bound slope for ambiguous neurons
# We optimize alpha to maximize the worst-case lower bound

def crown_with_alpha(W1, b1, W2, b2, x_lo, x_hi, alpha, output_idx=0):
    """CROWN lower bound with learnable alpha slopes."""
    n1 = len(b1)
    W1p = np.maximum(0, W1); W1n = np.minimum(0, W1)
    pre_lo = W1p @ x_lo + W1n @ x_hi + b1
    pre_hi = W1p @ x_hi + W1n @ x_lo + b1
    
    h_lo = np.zeros(n1)
    for i in range(n1):
        lo, hi = pre_lo[i], pre_hi[i]
        if lo >= 0:
            h_lo[i] = pre_lo[i]
        elif hi <= 0:
            h_lo[i] = 0.0
        else:
            # alpha[i] in [0, 1] controls the lower bound slope
            h_lo[i] = alpha[i] * pre_lo[i]  # pre_lo < 0 so h_lo_i >= 0 when alpha=0
    
    W2p = np.maximum(0, W2[output_idx])
    W2n = np.minimum(0, W2[output_idx])
    
    # Upper bound on hidden (for negative weights in output layer)
    h_hi = np.zeros(n1)
    for i in range(n1):
        lo, hi = pre_lo[i], pre_hi[i]
        if lo >= 0:
            h_hi[i] = pre_hi[i]
        elif hi <= 0:
            h_hi[i] = 0.0
        else:
            slope = hi / (hi - lo)
            h_hi[i] = slope * pre_hi[i] - slope * lo
    
    out_lo = W2p @ h_lo + W2n @ h_hi + b2[output_idx]
    return out_lo

# Optimize alpha for tightest lower bound
x_lo = x0 - 0.1; x_hi = x0 + 0.1
n_hidden = len(b1)

# Gradient ascent on alpha
alpha = np.zeros(n_hidden)  # start at 0
lr_alpha = 0.1

bounds_history = []
for step in range(50):
    lb = crown_with_alpha(W1, b1, W2, b2, x_lo, x_hi, alpha)
    bounds_history.append(lb)
    # Numerical gradient
    grad = np.zeros(n_hidden)
    for i in range(n_hidden):
        a_plus = alpha.copy(); a_plus[i] += 0.01
        a_minus = alpha.copy(); a_minus[i] -= 0.01
        grad[i] = (crown_with_alpha(W1, b1, W2, b2, x_lo, x_hi, a_plus) -
                   crown_with_alpha(W1, b1, W2, b2, x_lo, x_hi, a_minus)) / 0.02
    alpha = np.clip(alpha + lr_alpha * grad, 0, 1)

print('Alpha-CROWN Lower Bound Optimization:')
print(f'  Initial lower bound:   {bounds_history[0]:.4f}')
print(f'  Optimized lower bound: {bounds_history[-1]:.4f}')
print(f'  Improvement:           {bounds_history[-1] - bounds_history[0]:.4f}')
print(f'  Optimal alpha values:  {alpha.round(3)}')
print('\nTighter lower bound = stronger robustness certificate')
